In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv').fillna(0)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
df_train.drop(columns=['fuel_efficiency_mpg'], inplace=True)
df_val.drop(columns=['fuel_efficiency_mpg'], inplace=True)

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

# 1
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print( export_text(dt, feature_names=list(dv.get_feature_names_out())).split('\n')[0])

# 2
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
model = rf.fit(X_train, y_train)
y_pred = model.predict(X_val)
print(np.sqrt(mean_squared_error(y_val, y_pred)))
#3/4
params = [10, 50, 100, 150]
max_depth= [10, 15, 20, 25]
min_rmse = 1000000
best_params = []

for e in params:
    for md in max_depth:
        rf = RandomForestRegressor(n_estimators=e, max_depth=md, random_state=1, n_jobs=-1)
        model = rf.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        if rmse<min_rmse:
            min_rmse = rmse
            best_params = [e, md]
print("max_depth->", best_params[1],"e->", best_params[0],"RMSE->",min_rmse )

    

|--- model_year <= 1997.50
1.8332197358745623
max_depth-> 10 e-> 150 RMSE-> 1.7437162901130996


In [16]:
# 5

rf2 = RandomForestRegressor(n_estimators=10,
max_depth=20,
random_state=1,
n_jobs=-1)
model = rf2.fit(X_train, y_train)

print(model.feature_importances_)
print(dv.get_feature_names_out())



[0.05499763 0.00795063 0.02006223 0.00563848 0.06128144 0.01657053
 0.16071245 0.01627193 0.07827125 0.29610715 0.00731824 0.01940147
 0.00651495 0.00343644 0.04215174 0.20331344]
['acceleration' 'drivetrain=All-wheel drive'
 'drivetrain=Front-wheel drive' 'drivetrain=Rear-wheel drive'
 'engine_displacement' 'fuel_type=Diesel' 'fuel_type=Gasoline'
 'fuel_type=Hybrid' 'horsepower' 'model_year' 'num_cylinders' 'num_doors'
 'origin=Asia' 'origin=Europe' 'origin=USA' 'vehicle_weight']


In [19]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
for eta in [0.1, 0.3]:
    xgb_params = {
        'eta': eta, 
        'max_depth': 6,
        'min_child_weight': 1,
        
        'objective': 'reg:squarederror',
        'nthread': 8,
        
        'seed': 1,
        'verbosity': 1,
    }
    model2 = xgb.train(xgb_params, dtrain, num_boost_round=100)
    y_pred2 = model2.predict(dval)
    print(np.sqrt(mean_squared_error(y_val, y_pred2)))

1.724810059827976
1.8264579799904583
